# Notebook 10: Aggregate results

Consolidates result CSVs from Notebooks 04-09 into formatted tables and figures for the dissertation Results and Evaluation chapters. Reads written CSV artifacts; re-computes no underlying metrics. Outputs to `outputs/results/final` and `outputs/figures/final`.

## 1. Setup

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
importlib.reload(config)

config.ensure_output_dirs()
config.set_plot_style()

RESULTS = config.RESULTS_DIR
FINAL_R = RESULTS / "final"
FINAL_F = config.FIGURES_DIR / "final"

FINAL_R.mkdir(parents=True, exist_ok=True)
FINAL_F.mkdir(parents=True, exist_ok=True)

def load(filename):
    """Loads a CSV artifact from RESULTS_DIR if present, returning None if missing."""
    filepath = RESULTS / filename
    return pd.read_csv(filepath) if filepath.exists() else None


Mounted at /content/drive


## 2. Master table assembly

In [2]:
gap_table = load("05_transfer_gaps.csv")
run_level_auroc = load("05_run_level_auroc.csv")
run_level_ece = load("06_run_level_ece.csv")
debiased_ece = load("06_debiased_ece.csv")
subsampling_check = load("06_subsampling_check.csv")
size_table = load("07_size_scaling.csv")
sig = load("08_significance.csv")
fold_balance = load("02_fold_class_balance.csv")
musa_id = load("09_musa_id_generalization.csv")
k2m_summary = load("09_k2m_run_level.csv")
three_way = load("09_three_way_comparison.csv")

for name, df, out_name in [
    ("Transfer gaps (SQ1/SQ2 primary outcome)", gap_table, "table_1_transfer_gaps.csv"),
    ("Run-level AUROC by condition/architecture", run_level_auroc, "table_2_run_level_auroc.csv"),
    ("Run-level debiased ECE by condition/architecture", run_level_ece, "table_3_run_level_ece.csv"),
    ("Size-matched ECE subsampling check", subsampling_check, "table_4_subsampling_check.csv"),
    ("Size scaling (K2N curve vs N2K reference)", size_table, "table_5_size_scaling.csv"),
    ("Significance summary (Holm-corrected)", sig, "table_6_significance.csv"),
    ("Togunwa fold class balance diagnostic", fold_balance, "table_7_fold_balance.csv"),
    ("Musa generalization: per-run K2M AUROC/ECE (exploratory)", musa_id, "table_8_musa_k2m_per_run.csv"),
    ("Musa generalization: run-level K2M summary, per architecture", k2m_summary, "table_9_musa_k2m_summary.csv"),
    ("Musa generalization: K2K vs K2N vs K2M comparison (exploratory)", three_way, "table_10_musa_three_way_comparison.csv"),
]:
    if df is not None:
        df.to_csv(FINAL_R / out_name, index=False)
        print(f"\n{name}:")
        print(df.to_string(index=False))
    else:
        print(f"\n[Warning] {name} missing; re-run the corresponding notebook.")



Transfer gaps (SQ1/SQ2 primary outcome):
           arch  n_seeds_K  n_seeds_N  gap_K  gap_K_lo  gap_K_hi  gap_N  gap_N_lo  gap_N_hi  asymmetry  asym_lo  asym_hi
   mobilenet_v2          6          6  0.321     0.256     0.385  0.087     0.008     0.160      0.233    0.125    0.351
efficientnet_b0          6          6  0.377     0.301     0.435 -0.067    -0.105    -0.023      0.444    0.328    0.530

Run-level AUROC by condition/architecture:
condition            arch  n_runs  auroc_mean  auroc_lo  auroc_hi
      K2K efficientnet_b0      18       0.966     0.959     0.972
      K2N efficientnet_b0      18       0.588     0.550     0.633
 K2N_full efficientnet_b0      18       0.619     0.600     0.636
      N2K efficientnet_b0      30       0.730     0.683     0.776
      N2N efficientnet_b0      30       0.664     0.623     0.702
      K2K    mobilenet_v2      18       0.965     0.955     0.974
      K2N    mobilenet_v2      18       0.644     0.603     0.686
 K2N_full    mobilenet_

## 3. Figure assembly

In [3]:
figure_sources = [
    ("05_transfer_gaps.png",         "figure_1_transfer_gaps.png"),
    ("06_subsampling_check.png",     "figure_2_subsampling_check.png"),
    ("06_reliability.png",           "figure_3_reliability.png"),
    ("07_size_scaling.png",          "figure_4_size_scaling.png"),
    ("09_ood_confidence.png",        "figure_5_ood_confidence.png"),
    ("09_three_way_comparison.png",  "figure_6_musa_three_way_comparison.png"),
]
for src_name, dest_name in figure_sources:
    src = config.FIGURES_DIR / src_name
    if src.exists():
        shutil.copy(src, FINAL_F / dest_name)
        print(f"Copied {src_name} -> {dest_name}")
    else:
        print(f"[Note] {src_name} not found; run its notebook to include {dest_name}.")



Copied 05_transfer_gaps.png -> figure_1_transfer_gaps.png
Copied 06_subsampling_check.png -> figure_2_subsampling_check.png
Copied 06_reliability.png -> figure_3_reliability.png
Copied 07_size_scaling.png -> figure_4_size_scaling.png
Copied 09_ood_confidence.png -> figure_5_ood_confidence.png
Copied 09_three_way_comparison.png -> figure_6_musa_three_way_comparison.png


## 4. Plain-text results digest

In [4]:
lines = ["# Results Summary - crosspop-cxr-asymmetry", ""]

if gap_table is not None:
    lines.append("## Primary outcome: transfer gaps and asymmetry, per architecture")
    for _, r in gap_table.iterrows():
        lines.append(
            f"- {r['arch']}: Gap_K={r['gap_K']:.3f} [{r['gap_K_lo']:.3f},{r['gap_K_hi']:.3f}], "
            f"Gap_N={r['gap_N']:.3f} [{r['gap_N_lo']:.3f},{r['gap_N_hi']:.3f}], "
            f"Asymmetry={r['asymmetry']:.3f} [{r['asym_lo']:.3f},{r['asym_hi']:.3f}]"
        )
    lines.append("")

if run_level_ece is not None:
    lines.append("## Calibration: run-level debiased ECE by condition/architecture")
    for _, r in run_level_ece.iterrows():
        lines.append(
            f"- {r['arch']} / {r['condition']}: ECE={r['ece_debiased_mean']:.3f} "
            f"[{r['ece_lo']:.3f},{r['ece_hi']:.3f}] (n_runs={r['n_runs']})"
        )
    lines.append("")

if subsampling_check is not None:
    lines.append("## Size-matched subsampling check (does raw K2N-vs-N2K ECE gap survive size-matching?)")
    for _, r in subsampling_check.iterrows():
        verdict = "survives" if r["gap_survives_size_matching"] else "does NOT clearly survive (may be partly a small-n artifact)"
        lines.append(
            f"- {r['arch']}: K2N actual ECE={r['k2n_actual_ece']:.3f}, "
            f"N2K subsampled-to-30 mean={r['n2k_subsampled_mean_ece']:.3f} "
            f"[{r['n2k_subsampled_lo']:.3f},{r['n2k_subsampled_hi']:.3f}] -> gap {verdict}"
        )
    lines.append("Note: this checks the RAW cross-population ECE gap only. The primary calibration")
    lines.append("claim rests on the debiased, in-domain-baselined run-level table above, which")
    lines.append("compares each model's cross-population ECE against its OWN in-domain baseline")
    lines.append("(structurally matched in evaluation-set size pairwise), not on this raw comparison.")
    lines.append("")

if sig is not None:
    lines.append("## Significance summary (Holm-corrected where applicable)")
    for _, r in sig.iterrows():
        status = "significant/reliable" if r.get("significant") else "not significant/reliable"
        lines.append(f"- {r['test']}: estimate={r.get('estimate')}, p_holm={r.get('p_holm')} -> {status}")
    lines.append("")

if fold_balance is not None:
    max_dev = (fold_balance["train_ratio"] - 1.0).abs().max()
    lines.append(f"## Togunwa fold class balance: max deviation from 1:1 = {max_dev:.3f}")
    lines.append("")

if three_way is not None and k2m_summary is not None:
    lines.append("## Musa generalization check (exploratory, K2M direction only -- see caveats)")
    for _, r in three_way.iterrows():
        lines.append(f"- {r['arch']} / {r['condition']}: AUROC={r['auroc_mean']:.3f} [{r['auroc_lo']:.3f},{r['auroc_hi']:.3f}] (n_runs={r['n_runs']})")
    lines.append("")
    lines.append("Observations:")
    lines.append("- Cross-population degradation from K2K reproduces on Musa for both architectures,")
    lines.append("  supporting generalization of the core finding past a single dataset pair.")
    lines.append("- K2M AUROC is HIGHER than K2N (Togunwa) for both architectures, so the magnitude of")
    lines.append("  degradation is not consistent across cross-population targets -- dataset-specific")
    lines.append("  factors, not just \"cross-population-ness\" in the abstract, evidently matter.")
    lines.append("- K2M shows markedly higher per-run variability than K2N, especially for MobileNetV2")
    lines.append("  (individual run AUROCs range from 0.220 -- below-chance/inverted -- to 0.929),")
    lines.append("  giving K2M the widest CI of any condition in the study. This instability, not just")
    lines.append("  the interval width, is itself worth reporting.")
    lines.append("")
    lines.append("CAVEATS: (1) no reverse (Musa-trained) direction exists, so this is a single-direction")
    lines.append("descriptive check, NOT a formal Gap_Musa/asymmetry replication like the Togunwa result.")
    lines.append("(2) Musa cohort has mixed pediatric/adult composition (author confirmation, 2026),")
    lines.append("unlike Kermany/Togunwa (both pediatric-only); this confound is not isolated by this design.")
    lines.append("")

digest_content = "\n".join(lines)
digest_file = FINAL_R / "results_summary.md"
digest_file.write_text(digest_content)

print(f"\n{digest_content}")
print(f"\nSaved results summary digest to: {digest_file.name}")



# Results Summary - crosspop-cxr-asymmetry

## Primary outcome: transfer gaps and asymmetry, per architecture
- mobilenet_v2: Gap_K=0.321 [0.256,0.385], Gap_N=0.087 [0.008,0.160], Asymmetry=0.233 [0.125,0.351]
- efficientnet_b0: Gap_K=0.377 [0.301,0.435], Gap_N=-0.067 [-0.105,-0.023], Asymmetry=0.444 [0.328,0.530]

## Calibration: run-level debiased ECE by condition/architecture
- efficientnet_b0 / K2K: ECE=0.056 [0.039,0.076] (n_runs=18)
- efficientnet_b0 / K2N: ECE=0.245 [0.200,0.288] (n_runs=18)
- efficientnet_b0 / K2N_full: ECE=0.221 [0.200,0.241] (n_runs=18)
- efficientnet_b0 / N2K: ECE=0.155 [0.136,0.173] (n_runs=30)
- efficientnet_b0 / N2N: ECE=0.043 [0.025,0.066] (n_runs=30)
- mobilenet_v2 / K2K: ECE=0.085 [0.069,0.103] (n_runs=18)
- mobilenet_v2 / K2N: ECE=0.253 [0.215,0.293] (n_runs=18)
- mobilenet_v2 / K2N_full: ECE=0.263 [0.239,0.285] (n_runs=18)
- mobilenet_v2 / N2K: ECE=0.166 [0.137,0.196] (n_runs=30)
- mobilenet_v2 / N2N: ECE=0.092 [0.063,0.123] (n_runs=30)

## Size-mat

# 5. Summary Complete

In [5]:
print("\n=========================================================================")
print("                   PIPELINE EXECUTION COMPLETE                           ")
print("=========================================================================")
print(f"Master artifacts available in: {FINAL_R} and {FINAL_F}")



                   PIPELINE EXECUTION COMPLETE                           
Master artifacts available in: /content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry/outputs/results/final and /content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry/outputs/figures/final
